# Spatial TX registration to blockface

Process
1. Register spatial TX barcode to its respective blockface image
    a. If blockface section doesn't exist, pick the closets section  
    b. If blockface set doesn't exist, duplicate image from closest set and rename to target set. (See readme for mapping)  
    c. Perform QC on registration from the outputted QC images  
    d. For registration that failed, this is due to a failure in the initial rotation search. A rotation is visually picked and the registration is re-run


In [1]:
from hmba_stx_registration import Specimen, get_metadata_df, register_cells_to_blockface
from pathlib import Path

base_path = Path("/home/mike/workspace/data/macaque_brain/QM24.50.002")
sptx_path = base_path / "sptx"
barcodes_path = sptx_path / "xenium"
um_per_px = 20
mm_per_px = um_per_px / 1000
metadata_df = get_metadata_df(barcodes_path)
mapping_date = "20260528"
output_date = "20260610"

/home/mike/workspace/data/macaque_brain/QM24.50.002/sptx/xenium/blockface_images/blockface_images_specimen_metadata.json does not exist, skipping
/home/mike/workspace/data/macaque_brain/QM24.50.002/sptx/xenium/QM24.50.002_section_metadata.csv is not a directory, skipping
/home/mike/workspace/data/macaque_brain/QM24.50.002/sptx/xenium/registration_plots/registration_plots_specimen_metadata.json does not exist, skipping
/home/mike/workspace/data/macaque_brain/QM24.50.002/sptx/xenium/QM24.50.002_section_metadata_20260305.csv is not a directory, skipping
/home/mike/workspace/data/macaque_brain/QM24.50.002/sptx/xenium/barcode_csv/barcode_csv_specimen_metadata.json does not exist, skipping


In [2]:
import numpy as np
barcodes = metadata_df['barcode'].tolist()
processed_blocks = []
for barcode in barcodes:
    specimen = Specimen(barcode, barcodes_path)
    processed_blocks.append(".".join(specimen.specimen_set_name.split(".")[:-1]))
processed_blocks = np.unique(processed_blocks)
print(processed_blocks)

['QM24.50.002.CX.41.01' 'QM24.50.002.CX.41.02' 'QM24.50.002.CX.42.01'
 'QM24.50.002.CX.42.02' 'QM24.50.002.CX.43.01' 'QM24.50.002.CX.43.02'
 'QM24.50.002.CX.43.03' 'QM24.50.002.CX.44.01' 'QM24.50.002.CX.44.02'
 'QM24.50.002.CX.44.03' 'QM24.50.002.CX.44.04' 'QM24.50.002.CX.45.01'
 'QM24.50.002.CX.45.02' 'QM24.50.002.CX.45.03' 'QM24.50.002.CX.45.04'
 'QM24.50.002.CX.45.05' 'QM24.50.002.CX.46.01' 'QM24.50.002.CX.46.02'
 'QM24.50.002.CX.46.03' 'QM24.50.002.CX.46.04' 'QM24.50.002.CX.46.05'
 'QM24.50.002.CX.46.06' 'QM24.50.002.CX.47.01' 'QM24.50.002.CX.47.02'
 'QM24.50.002.CX.47.03' 'QM24.50.002.CX.47.04' 'QM24.50.002.CX.47.05'
 'QM24.50.002.CX.47.06' 'QM24.50.002.CX.47.07' 'QM24.50.002.CX.48.01'
 'QM24.50.002.CX.48.02' 'QM24.50.002.CX.48.03' 'QM24.50.002.CX.48.04'
 'QM24.50.002.CX.48.05' 'QM24.50.002.CX.48.06' 'QM24.50.002.CX.48.07'
 'QM24.50.002.CX.49.01' 'QM24.50.002.CX.49.02' 'QM24.50.002.CX.49.03'
 'QM24.50.002.CX.49.04' 'QM24.50.002.CX.49.05' 'QM24.50.002.CX.49.06'
 'QM24.50.002.CX.49.

### Perform registration between ST and Blockface

For each barcode:
-> Register to blockface  
**Inputs:**  
1. blockface image path  
2. Specimen object  
    a. cells table  
    b. specimen metadata  

**Outputs:**  
1. transforms_out_path: directory to store   affines in .npy format  
2. qc_path: directory to store plots to visualize registration for QC  


In [3]:
barcodes = metadata_df['barcode'].tolist()
transforms_path_existing = sptx_path / "sptx_to_bf_transforms"
existing_specimens = [p.stem.split("_")[0] for p in transforms_path_existing.glob("*.npy")]
transforms_path = sptx_path / f"st2block_output_{output_date}"
existing_specimens.extend([p.stem.split("_")[0] for p in transforms_path.glob("*.npy")])
for barcode in barcodes:
    specimen = Specimen(barcode, barcodes_path, date=mapping_date)
    if specimen.specimen_name in existing_specimens:
        print(f"Specimen {specimen.specimen_name} already has a transform, skipping.")
        continue
    print("Registering specimen:", specimen.specimen_name)
    bf_img_path_base = base_path / 'blocks' / specimen.slab_name / 'processed'
    try:
        bf_img_path = list(bf_img_path_base.glob(f"{specimen.specimen_set_name}*.png"))[0]
    except IndexError:
        print(f"No blockface image found for specimen {specimen.specimen_name}, skipping.")
        continue
    st2bf_affine = register_cells_to_blockface(specimen.cells_table,
                            bf_img_path,
                            specimen.specimen_name,
                            transforms_out_path=transforms_path,
                            qc_path=transforms_path / "qc",
                            table_label="subclass_label",
                            gamma=0.3
                            )

Specimen QM24.50.002.CX.45.05.01.02 already has a transform, skipping.
Specimen QM24.50.002.CX.52.01.03.02 already has a transform, skipping.
Specimen QM24.50.002.CX.49.08.01.02 already has a transform, skipping.
Registering specimen: QM24.50.002.CX.53.06.01.02


Cells table contains 1966 rows with NaN center_x or center_y. These will be dropped for downstream processing.


Specimen QM24.50.002.CX.47.01.05.02 already has a transform, skipping.
Specimen QM24.50.002.CX.48.02.01.02 already has a transform, skipping.
Specimen QM24.50.002.CX.48.06.03.02 already has a transform, skipping.
Specimen QM24.50.002.CX.47.06.01.02 already has a transform, skipping.
Specimen QM24.50.002.CX.41.01.01.02 already has a transform, skipping.
Specimen QM24.50.002.CX.50.03.03.02 already has a transform, skipping.
Specimen QM24.50.002.CX.48.02.05.03 already has a transform, skipping.
Specimen QM24.50.002.CX.51.06.03.02 already has a transform, skipping.
Specimen QM24.50.002.CX.45.01.01.02 already has a transform, skipping.
Specimen QM24.50.002.CX.46.01.05.02 already has a transform, skipping.
Specimen QM24.50.002.CX.46.05.05.02 already has a transform, skipping.
Specimen QM24.50.002.CX.52.01.05.02 already has a transform, skipping.
Registering specimen: QM24.50.002.CX.53.05.01.02


Cells table contains 770 rows with NaN center_x or center_y. These will be dropped for downstream processing.


Specimen QM24.50.002.CX.47.02.04.02 already has a transform, skipping.
Registering specimen: QM24.50.002.CX.54.01.01.02
No blockface image found for specimen QM24.50.002.CX.54.01.01.02, skipping.
Specimen QM24.50.002.CX.45.01.03.02 already has a transform, skipping.
Specimen QM24.50.002.CX.43.02.03.02 already has a transform, skipping.
Specimen QM24.50.002.CX.45.05.03.02 already has a transform, skipping.
Specimen QM24.50.002.CX.44.02.03.02 already has a transform, skipping.
Specimen QM24.50.002.CX.51.01.07.03 already has a transform, skipping.
Specimen QM24.50.002.CX.52.02.03.02 already has a transform, skipping.
Specimen QM24.50.002.CX.46.04.05.02 already has a transform, skipping.
Registering specimen: QM24.50.002.CX.53.07.05.02


...

Specimen QM24.50.002.CX.51.05.05.02 already has a transform, skipping.
Specimen QM24.50.002.CX.46.05.03.02 already has a transform, skipping.
Specimen QM24.50.002.CX.48.02.03.02 already has a transform, skipping.
Specimen QM24.50.002.CX.50.01.04.02 already has a transform, skipping.
Specimen QM24.50.002.CX.48.05.05.03 already has a transform, skipping.
Specimen QM24.50.002.CX.48.01.01.02 already has a transform, skipping.
Specimen QM24.50.002.CX.50.07.01.02 already has a transform, skipping.
Specimen QM24.50.002.CX.47.04.05.02 already has a transform, skipping.
Specimen QM24.50.002.CX.45.03.03.02 already has a transform, skipping.
Specimen QM24.50.002.CX.50.04.01.03 already has a transform, skipping.
Registering specimen: QM24.50.002.CX.53.05.03.02


Cells table contains 1044 rows with NaN center_x or center_y. These will be dropped for downstream processing.


Registering specimen: QM24.50.002.CX.54.01.03.02
No blockface image found for specimen QM24.50.002.CX.54.01.03.02, skipping.
Specimen QM24.50.002.CX.49.07.03.02 already has a transform, skipping.
Specimen QM24.50.002.CX.50.01.01.02 already has a transform, skipping.
Specimen QM24.50.002.CX.47.03.05.02 already has a transform, skipping.
Registering specimen: QM24.50.002.CX.53.07.01.02


Cells table contains 56 rows with NaN center_x or center_y. These will be dropped for downstream processing.


Specimen QM24.50.002.CX.46.06.01.02 already has a transform, skipping.
Specimen QM24.50.002.CX.47.02.02.02 already has a transform, skipping.
Specimen QM24.50.002.CX.50.04.03.01 already has a transform, skipping.


In [4]:
list(specimen.cells_table["cl_label"].unique())

['macaque:NN-IMN:343',
 'macaque:NN-IMN:155',
 'macaque:Pallium-Glut:220',
 'macaque:Subpallium-GABA:51',
 'macaque:Pallium-Glut:111',
 'macaque:Subpallium-GABA:277',
 'macaque:NN-IMN:191',
 'macaque:Subpallium-GABA:140',
 'macaque:Subpallium-GABA:328',
 'macaque:NN-IMN:62',
 'macaque:Subpallium-GABA:355',
 'macaque:Pallium-Glut:265',
 'macaque:NN-IMN:68',
 'macaque:NN-IMN:228',
 'macaque:NN-IMN:27',
 'macaque:Subpallium-GABA:132',
 'macaque:Subpallium-GABA:367',
 'macaque:Subpallium-GABA:426',
 'macaque:NN-IMN:208',
 'macaque:NN-IMN:307',
 'macaque:Pallium-Glut:145',
 'macaque:Subpallium-GABA:273',
 'macaque:NN-IMN:163',
 'macaque:NN-IMN:249',
 'macaque:NN-IMN:22',
 'macaque:Pallium-Glut:99',
 'macaque:TH-EPI-Glut:28',
 'macaque:Subpallium-GABA:344',
 'macaque:Subpallium-GABA:146',
 'macaque:Subpallium-GABA:352',
 'macaque:NN-IMN:344',
 'macaque:NN-IMN:51',
 'macaque:Subpallium-GABA:317',
 'macaque:NN-IMN:21',
 'macaque:Pallium-Glut:263',
 'macaque:NN-IMN:16',
 'macaque:NN-IMN:340',
 

In [7]:
specimen.table_label

'neighborhood_label'

In [6]:
import pandas as pd
specimen.cells_table.loc[pd.isna(specimen.cells_table["center_x"])]

,center_x,center_y,z,volume,area,polygon_center_x,polygon_center_y,polygon_center_z,SpotTable_cell_id,cell_label,...,cl_label,cl_name,cl_alias,cl_bootstrapping_probability,cl_aggregate_probability,cl_correlation_coefficient,neighborhood_entropy,class_entropy,subclass_entropy,cl_entropy
1491882309_aadieeij-1,NaN,NaN,NaN,NaN,43.033908,2395.608643,2429.825928,NaN,99621,1491882309_aadieeij-1,...,macaque:HY-EA-Glut-GABA:1,macaque:HY-EA-Glut-GABA:1,macaque:HY-EA-Glut-GABA:1,1.0,1.0,0.0,0.0,0.0,0.0,0.0
1491882309_aalpdpbg-1,NaN,NaN,NaN,NaN,70.263128,1072.217285,4885.945801,NaN,99622,1491882309_aalpdpbg-1,...,macaque:HY-EA-Glut-GABA:1,macaque:HY-EA-Glut-GABA:1,macaque:HY-EA-Glut-GABA:1,1.0,1.0,0.0,0.0,0.0,0.0,0.0
1491882309_abaeioam-1,NaN,NaN,NaN,NaN,39.827814,2623.133301,11640.052734,NaN,99623,1491882309_abaeioam-1,...,macaque:HY-EA-Glut-GABA:1,macaque:HY-EA-Glut-GABA:1,macaque:HY-EA-Glut-GABA:1,1.0,1.0,0.0,0.0,0.0,0.0,0.0
1491882309_abajhbia-1,NaN,NaN,NaN,NaN,83.448753,655.782104,10518.365234,NaN,99624,1491882309_abajhbia-1,...,macaque:HY-EA-Glut-GABA:1,macaque:HY-EA-Glut-GABA:1,macaque:HY-EA-Glut-GABA:1,1.0,1.0,0.0,0.0,0.0,0.0,0.0
1491882309_abbogihk-1,NaN,NaN,NaN,NaN,42.492033,8222.064453,7902.856934,NaN,99625,1491882309_abbogihk-1,...,macaque:HY-EA-Glut-GABA:1,macaque:HY-EA-Glut-GABA:1,macaque:HY-EA-Glut-GABA:1,1.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1491882309_oihkdoff-1,NaN,NaN,NaN,NaN,35.086408,2225.061523,12418.045898,NaN,100070,1491882309_oihkdoff-1,...,macaque:HY-EA-Glut-GABA:1,macaque:HY-EA-Glut-GABA:1,macaque:HY-EA-Glut-GABA:1,1.0,1.0,0.0,0.0,0.0,0.0,0.0
1491882309_oihlfjme-1,NaN,NaN,NaN,NaN,210.382976,2276.884766,12384.549805,NaN,100071,1491882309_oihlfjme-1,...,macaque:HY-EA-Glut-GABA:1,macaque:HY-EA-Glut-GABA:1,macaque:HY-EA-Glut-GABA:1,1.0,1.0,0.0,0.0,0.0,0.0,0.0
1491882309_oihncjmb-1,NaN,NaN,NaN,NaN,264.705947,2238.225342,12365.459961,NaN,100072,1491882309_oihncjmb-1,...,macaque:HY-EA-Glut-GABA:1,macaque:HY-EA-Glut-GABA:1,macaque:HY-EA-Glut-GABA:1,1.0,1.0,0.0,0.0,0.0,0.0,0.0
1491882309_oilemnlb-1,NaN,NaN,NaN,NaN,14.720938,1741.785522,10684.406250,NaN,100073,1491882309_oilemnlb-1,...,macaque:HY-EA-Glut-GABA:1,macaque:HY-EA-Glut-GABA:1,macaque:HY-EA-Glut-GABA:1,1.0,1.0,0.0,0.0,0.0,0.0,0.0


### Rerun registration for sets that failed QC by visually defining the rotation

In [ ]:
# setname_rotation = {
#  'QM24.50.002.CX.47.04.01': 180,
#  'QM24.50.002.CX.48.05.01': 180,
#  'QM24.50.002.CX.48.04.03': 180,
#  'QM24.50.002.CX.45.05.03': 45,  
#  'QM24.50.002.CX.47.03.03': 180,
#  'QM24.50.002.CX.49.07.03': 0
#  'QM24.50.002.CX.50.03.01': 180,
#  'QM24.50.002.CX.50.01.01': 180,
#  'QM24.50.002.CX.50.05.03': 0,
#  'QM24.50.002.CX.50.08.01': 90,
#  'QM24.50.002.CX.50.08.03': 110,
#  'QM24.50.002.CX.50.08.05': 135,
#  'QM24.50.002.CX.50.09.03': -170,
#  'QM24.50.002.CX.51.01.07': 180,
#  'QM24.50.002.CX.47.02.01': 180,
#  'QM24.50.002.CX.47.04.03': 180,
# }

setname_rotation = {
    'QM24.50.002.CX.51.09.01': -15,
    'QM24.50.002.CX.52.06.05': 0,
    'QM24.50.002.CX.52.02.05': 0,
    'QM24.50.002.CX.51.04.01': 0,
    'QM24.50.002.CX.51.04.03': 0,
    'QM24.50.002.CX.51.03.01': 0,
    'QM24.50.002.CX.52.06.01': 0,
    'QM24.50.002.CX.52.05.03': 0,
    'QM24.50.002.CX.52.02.01': 0,
    
}


transforms_path = sptx_path / f"st2block_output_{output_date}"
for setname, rotation in setname_rotation.items():
    for barcode in metadata_df.loc[metadata_df.specimen_set_name == setname].barcode.values:
            specimen = Specimen(barcode, barcodes_path, date=mapping_date)
            print("Registering specimen:", specimen.specimen_name)
            bf_img_path_base = base_path / 'blocks' / specimen.slab_name / 'processed'
            bf_img_path = list(bf_img_path_base.glob(f"{specimen.specimen_set_name}*.png"))[0]

            st2bf_affine = register_cells_to_blockface(specimen.cells_table,
                                    bf_img_path,
                                    specimen.specimen_name,
                                    transforms_out_path=transforms_path,
                                    qc_path=transforms_path / "qc",
                                    gamma=0.3,
                                    rotation=rotation,
                                    table_label="subclass_label",
                                    )


Registering specimen: QM24.50.002.CX.51.04.03.02


Cells table contains 364 rows with NaN center_x or center_y. These will be dropped for downstream processing.
